In [17]:
!pip install wandb calflops torch torchvision

In [18]:
! pip install wandb

In [26]:
import wandb

wandb.login()

True

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from calflops import calculate_flops
import wandb
import os

# --- 1. CLEAN LOGIN ---
os.environ.pop('WANDB_API_KEY', None)
# PASTE YOUR NEW KEY HERE
wandb.login(key=WANDB_API_KEY)

# --- 2. CUSTOM DATASET (Requirement #3) ---
class MyCIFAR10Dataset(Dataset):
    def __init__(self, train=True):
        self.data = datasets.CIFAR10(root='./data', train=train, download=True,
                                     transform=transforms.Compose([
                                         transforms.ToTensor(),
                                         transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
                                     ]))
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

# --- 3. CNN MODEL (Requirement #1 & #2) ---
class Lab2CNN(nn.Module):
    def __init__(self):
        super(Lab2CNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 512), nn.ReLU(),
            nn.Linear(512, 10)
        )
    def forward(self, x): return self.classifier(self.features(x))

# --- 4. WANDB INIT (Requirement #8) ---
# Using your new personal ID 'b23bb1020'
run = wandb.init(
    # entity="b23bb1020",
    project="Lab2_Final_Submission",
    config={"lr": 0.001, "epochs": 25, "batch_size": 64}
)

wandb.config.update({"visibility": "public"})

# --- 5. FLOPs & WATCH (Requirement #4 & #6) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Lab2CNN().to(device)
flops, macs, params = calculate_flops(model=model, input_shape=(1, 3, 32, 32), print_results=False)
wandb.config.update({"total_flops": flops, "params": params})
wandb.watch(model, log="all", log_freq=100) # This captures weight/gradient flow

# --- 6. TRAINING (Requirement #5) ---
train_loader = DataLoader(MyCIFAR10Dataset(train=True), batch_size=64, shuffle=True)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print("Training started on NEW account...")
for epoch in range(25):
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        loss = criterion(model(data), target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            run.log({"loss": loss.item(), "epoch": epoch + 1})

run.finish()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇██
loss,█▅▅▄▄▃▃▃▃▃▄▃▃▂▂▃▂▃▂▂▂▂▂▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
epoch,10
loss,0.02179


Training started on NEW account...


epoch,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇██
loss,█▆▅▆▅▄▃▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▂▂▁▃▁▁▁▁▁▁▁▂▁▁▁▁▁
epoch,25
loss,0.0021
